## Commit changes

In [10]:
!cd ~/work/python-ai-learning && git add 01-environnement/02-conda.ipynb 
git commit -m 'Completer notebook conda avec outputs' && git push origin master

[Errno 2] No such file or directory: '/home/bbrisson/work/python-ai-learning && git add 01-environnement/02-conda.ipynb && git commit -m Completer notebook conda avec outputs && git push origin master'
/home/bbrisson/work/python-ai-learning/01-environnement


# Conda — Gestion d'environnements

Ce notebook s'apprend en **exécutant chaque cellule soi-même** avec Shift+Enter.

### Exécuter des commandes conda dans Jupyter

Le `!` devant une commande l'exécute dans le shell. Cependant, JupyterLab sur le GX10 tourne dans son propre environnement virtuel (`.venv`, Python 3.12) — **indépendant de conda**. Le PATH de ce processus ne contient pas conda.

```
JupyterLab (.venv Python 3.12)   ← ne connaît pas conda
    └── kernel ai_learning        ← connaît Python 3.11 de conda
```

### Trois approches possibles

**1. Chemin absolu** — solution pragmatique retenue dans ce notebook :
```python
!/home/bbrisson/miniconda3/bin/conda env list
```

**2. `python -m conda`** — fonctionne uniquement si le `python` appelé est celui qui possède conda. Ce n'est pas le cas ici : le kernel `ai_learning` est géré *par* conda, mais ne le contient pas.

**3. `subprocess`** — l'équivalent Python pur du chemin absolu :
```python
import subprocess
result = subprocess.run(
    ["/home/bbrisson/miniconda3/bin/conda", "env", "list"],
    capture_output=True, text=True
)
print(result.stdout)
```

**La solution vraiment propre** serait de configurer le PATH du serveur JupyterLab pour inclure conda au démarrage — mais c'est la configuration du GX10 qu'on ne contrôle pas. Le chemin absolu est donc notre solution dans ce contexte.

 ## 1. Pourquoi des environnements ?

Imagine deux projets sur la même machine :

| Projet | Librairie | Version requise |
|--------|-----------|----------------|
| Projet A | TensorFlow | 2.10 |
| Projet B | TensorFlow | 2.15 |

Sans environnements, impossible d'avoir les deux en même temps — une version écrase l'autre.

**Analogie C++ :** c'est le même problème que d'avoir deux projets qui requièrent des versions incompatibles d'une même `.so`. La solution en C++ c'est de compiler chaque projet avec ses propres librairies. En Python, la solution c'est les environnements.

Un environnement = un dossier isolé contenant :
- Un interpréteur Python d'une version spécifique
- Un ensemble de paquets dans des versions spécifiques
- Des variables d'environnement propres

In [ ]:
!/home/bbrisson/miniconda3/bin/conda --version

## 2. Lister les environnements existants

In [ ]:
!/home/bbrisson/miniconda3/bin/conda env list

**Ce que tu dois voir :**
- `base` — l'environnement de base installé avec Miniconda
- `ai_learning` — notre environnement de travail
- D'autres environnements existants sur la machine

L'astérisque `*` indique l'environnement **actif** au moment de l'exécution.

## 3. Inspecter un environnement

Pour voir ce qui est installé dans `ai_learning` :

In [ ]:
!/home/bbrisson/miniconda3/bin/conda list -n ai_learning

**Ce que tu dois voir :** la liste de tous les paquets installés avec leur version et leur canal source (`defaults`, `conda-forge`, etc.).

Note la colonne **Build** — conda gère non seulement la version Python du paquet, mais aussi la version compilée (CPU, GPU, architecture ARM64, etc.).

## 4. Créer un environnement

La commande suivante crée un environnement de test — on va l'utiliser pour pratiquer, puis le supprimer.

**Décryptage :**
- `-n test_env` : nom de l'environnement
- `python=3.11` : version de Python souhaitée
- `-y` : confirme automatiquement sans demander

**Ce que tu vas voir :** conda résout les dépendances, télécharge et installe les paquets. Ça peut prendre 1-2 minutes.

In [ ]:
!/home/bbrisson/miniconda3/bin/conda create -n test_env python=3.11 -y

## 5. Installer un paquet dans un environnement

Il y a deux façons d'installer un paquet dans un environnement :

**Méthode A — depuis l'extérieur** (ce qu'on utilisera dans Jupyter) :
```bash
conda install -n nom_env paquet -y
```

**Méthode B — depuis l'intérieur** (en ligne de commande, après activation) :
```bash
conda activate nom_env
conda install paquet -y
```

Dans Jupyter on utilise toujours la **Méthode A** car on ne peut pas faire `conda activate` dans une cellule bash — l'activation ne persiste pas entre les cellules.

Installons `requests` dans notre environnement de test :

In [3]:
!/home/bbrisson/miniconda3/bin/conda install -n test_env requests -y

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - defaults
Platform: linux-aarch64
Solving environment: done

# All requested packages already installed.



## 6. conda vs pip — quand utiliser lequel ?

| | conda | pip |
|--|-------|-----|
| **Gère** | Python + dépendances natives (C, CUDA, etc.) | Python uniquement |
| **Vérifie** | Compatibilité entre tous les paquets | Pas de vérification globale |
| **Vitesse** | Plus lent | Plus rapide |
| **Paquets disponibles** | Moins (mais les essentiels AI/ML y sont) | Plus (tout PyPI) |

**Règle pratique :**
1. Toujours essayer `conda install` en premier
2. Si le paquet n'est pas disponible dans conda → utiliser `pip install`
3. Ne jamais mélanger conda et pip sans précaution dans le même environnement

**Pourquoi la règle 3 ?** Conda et pip ont des solveurs de dépendances indépendants. Pip peut installer quelque chose qui casse la cohérence que conda avait établie, sans s'en rendre compte.

## 7. Supprimer un environnement

On supprime l'environnement de test :

In [1]:
!/home/bbrisson/miniconda3/bin/conda env remove -n test_env -y

Jupyter detected...
2 channel Terms of Service accepted

Remove all packages in environment /home/bbrisson/miniconda3/envs/test_env:


## Package Plan ##

  environment location: /home/bbrisson/miniconda3/envs/test_env


The following packages will be REMOVED:

  _libgcc_mutex-0.1-main
  _openmp_mutex-5.1-51_gnu
  brotlicffi-1.2.0.0-py311h5b11e9f_0
  bzip2-1.0.8-h998d150_6
  ca-certificates-2025.12.2-hd43f75c_0
  certifi-2026.01.04-py311hd43f75c_0
  cffi-2.0.0-py311h2d8c4a8_1
  charset-normalizer-3.4.4-py311hd43f75c_0
  idna-3.11-py311hd43f75c_0
  ld_impl_linux-aarch64-2.44-h97084ae_3
  libexpat-2.7.5-h5b11e9f_0
  libffi-3.4.4-h419075a_1
  libgcc-15.2.0-hc18542e_7
  libgcc-ng-15.2.0-h51576c1_7
  libgomp-15.2.0-hf47c802_7
  libnsl-2.0.0-h998d150_0
  libstdcxx-15.2.0-h9538471_7
  libstdcxx-ng-15.2.0-h719ae8e_7
  libuuid-1.41.5-h998d150_0
  libxcb-1.17.0-hf66535e_0
  libzlib-1.3.1-h998d150_0
  ncurses-6.5-h419075a_0
  openssl-3.5.5-h39c9139_0
  packaging-25.0-py311hd43f75c_1
  pip-26.0.1-

In [2]:
!/home/bbrisson/miniconda3/bin/conda env list


# conda environments:
#
# * -> active
# + -> frozen
base                     /home/bbrisson/miniconda3
ai_learning              /home/bbrisson/miniconda3/envs/ai_learning
facefusion               /home/bbrisson/miniconda3/envs/facefusion
facefusion310            /home/bbrisson/miniconda3/envs/facefusion310



`test_env` ne devrait plus apparaître dans la liste.

## Référence rapide

Dans ce notebook, `conda` désigne `/home/bbrisson/miniconda3/bin/conda`.

```bash
# Lister les environnements
!conda env list

# Créer un environnement
!conda create -n nom_env python=3.11 -y

# Lister les paquets d'un environnement
!conda list -n nom_env

# Installer un paquet
!conda install -n nom_env paquet -y

# Supprimer un environnement
!conda env remove -n nom_env -y
```